In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Eng.Shital\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [3]:
df = pd.read_csv('IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [17]:
# Data Understanding

print(df.shape)
print(df['sentiment'].value_counts())

df.sample(5)

(50000, 3)
sentiment
1    25000
0    25000
Name: count, dtype: int64


,review,sentiment,clean_text
44230,You would probably get something like this. I'...,0,would probabl get someth like translat movi li...
6533,This is the most stupid movie ever made. The s...,0,stupid movi ever made stori laughabl wife kid ...
21657,"...there was ""Broadcast News,"" and what a good...",1,broadcast news good thing one plain stand soun...
25894,I bought this game on eBay having heard that i...,1,bought game ebay heard similar game elit gamep...
25185,"When I saw this movie, I was amazed that it wa...",1,saw movi amaz tv movi think movi theater seen ...


In [5]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

In [18]:
# NLP Preprocessing

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()  # lowercasing
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # remove special chars
    
    words = text.split()  # tokenization
    
    words = [w for w in words if w not in stop_words]  # remove stopwords
    words = [stemmer.stem(w) for w in words]  # stemming
    
    return " ".join(words)

In [7]:
df['clean_text'] = df['review'].apply(preprocess)

In [19]:
# Feature Engineering

bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_text']).toarray()

In [9]:
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_text']).toarray()

In [10]:
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

In [11]:
# Model Building

lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [12]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [13]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

In [14]:
# Evaluation Function
def evaluate(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))

In [15]:
print("Logistic Regression")
evaluate(y_test, y_pred_lr)

print("\nNaive Bayes")
evaluate(y_test, y_pred_nb)

print("\nDecision Tree")
evaluate(y_test, y_pred_dt)

Logistic Regression
Accuracy: 0.885
Precision: 0.8750241080038573
Recall: 0.9003770589402659
F1 Score: 0.8875195618153364

Naive Bayes
Accuracy: 0.8518
Precision: 0.8483839373163565
Recall: 0.8594959317324866
F1 Score: 0.853903785488959

Decision Tree
Accuracy: 0.7131
Precision: 0.7175220529270249
Recall: 0.7102599722167097
F1 Score: 0.7138725441308467


In [16]:
# Comparison Table

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    'F1 Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_dt)
    ]
})

results

,Model,Accuracy,F1 Score
0,Logistic Regression,0.8850,0.887520
1,Naive Bayes,0.8518,0.853904
2,Decision Tree,0.7131,0.713873


Conclusion:

- Logistic Regression performed best because it handles sparse data well.
- TF-IDF gave better results than Bag of Words.
- Naive Bayes was fast but slightly less accurate.
- Decision Tree overfitted the data.

Final Choice: Logistic Regression + TF-IDF